# Quantum Error Correction: 3-Qubit Bit-Flip Code

## Goal
This notebook demonstrates a basic quantum error correction (QEC) protocol:
the 3-qubit bit-flip code.

We will:
- Encode a logical qubit into 3 physical qubits
- Introduce a bit-flip error
- Detect the error using syndrome measurements
- Analyze results using a simulator

## Why this matters
Quantum computers are highly sensitive to noise:
- decoherence
- gate errors
- measurement errors

Quantum error correction allows us to protect information despite these issues.

## Theoretical background: How and Why the 3-Qubit Bit-Flip Code Works

---

### 1. The Core Problem: Why Can't We Just Copy Qubits?

In classical computing, protecting a bit is easy — just copy it three times.  
If one copy flips, the majority vote wins: `0,0,1` → the correct value is `0`.

In quantum computing, **we cannot copy an unknown qubit**. This is the **No-Cloning Theorem**:

> *There is no quantum operation that can produce two identical copies of an arbitrary unknown quantum state.*

Formally, there is no unitary $U$ such that:
$$U(|\psi\rangle \otimes |0\rangle) = |\psi\rangle \otimes |\psi\rangle \quad \text{for all } |\psi\rangle$$

This seems to make error correction impossible. The 3-qubit code is clever because it **does not copy the qubit** — it *entangles* it across three qubits in a way that still allows error detection.

---

### 2. The Encoding: Entanglement, Not Copying

We want to protect a general qubit:
$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$$

The encoding maps this to a **3-qubit entangled state**:
$$|\psi_L\rangle = \alpha|000\rangle + \beta|111\rangle$$

This is achieved with two CNOT gates:

```
q0: ──■────■──   (control)
q1: ──X────┼──   (target of first CNOT)
q2: ────── X──   (target of second CNOT)
```

**Step by step:**
- Start: $(\alpha|0\rangle + \beta|1\rangle) \otimes |0\rangle \otimes |0\rangle = \alpha|000\rangle + \beta|100\rangle$
- After CNOT(q0→q1): $\alpha|000\rangle + \beta|110\rangle$
- After CNOT(q0→q2): $\alpha|000\rangle + \beta|111\rangle$ ✓

This is **not** three independent copies of $|\psi\rangle$. The three qubits are **entangled** — they share quantum correlations. The coefficients $\alpha$ and $\beta$ are stored in the *relationship* between qubits, not in any single one.

---

### 3. The Error Model: What Does a Bit-Flip Look Like Mathematically?

A bit-flip error on qubit $i$ is the action of the **Pauli-X gate** on that qubit:
$$X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}, \quad X|0\rangle = |1\rangle, \quad X|1\rangle = |0\rangle$$

Applied to the encoded state, errors produce distinguishable "error states":

| Error | Resulting state |
|-------|----------------|
| None | $\alpha|000\rangle + \beta|111\rangle$ |
| $X_0$ | $\alpha|100\rangle + \beta|011\rangle$ |
| $X_1$ | $\alpha|010\rangle + \beta|101\rangle$ |
| $X_2$ | $\alpha|001\rangle + \beta|110\rangle$ |

These four states are **mutually orthogonal** — they live in different subspaces of the 3-qubit Hilbert space. This is the key: we can distinguish between them without ever learning $\alpha$ or $\beta$.

---

### 4. Syndrome Measurement: The Magic of Not Looking at the Data

This is the most subtle and important part of the code.

**The fundamental challenge**: measuring qubits collapses superpositions. If we measured q0, q1, q2 directly, we would destroy the encoded $\alpha$ and $\beta$ — the very information we are trying to protect.

**The solution**: we measure *parity* — not individual qubit values.

We use two ancilla qubits (a0, a1) initialized to $|0\rangle$ and perform:

- **Syndrome bit 1** = parity of (q0, q1): `CNOT(q0→a0)`, `CNOT(q1→a0)` → reveals whether q0 and q1 agree  
- **Syndrome bit 2** = parity of (q1, q2): `CNOT(q1→a1)`, `CNOT(q2→a1)` → reveals whether q1 and q2 agree  

The ancilla qubit a0 ends up in state $|q_0 \oplus q_1\rangle$ (XOR of the two data qubits). Measuring it gives `0` if they agree, `1` if they differ — without ever revealing the individual values.

**Why this is safe**: These parity measurements commute with the encoded logical information, so measuring them does not disturb $\alpha$ or $\beta$. Formally, the parity operators $Z_0 Z_1$ and $Z_1 Z_2$ commute with the logical operators $\bar{X} = X_0 X_1 X_2$ and $\bar{Z} = Z_0$.

---

### 5. Decoding the Syndrome Table — With Reasoning

After measuring the two ancilla qubits we get a 2-bit syndrome $(s_1, s_0)$. Here is *why* each outcome maps to each error:

| Syndrome $(s_1, s_0)$ | q0=q1? | q1=q2? | Diagnosis | Fix |
|----------------------|--------|--------|-----------|-----|
| `00` | ✓ agree | ✓ agree | No error | Do nothing |
| `01` | ✗ differ | ✓ agree | q0 is the odd one out | Apply $X_0$ |
| `11` | ✗ differ | ✗ differ | q1 disagrees with both neighbours | Apply $X_1$ |
| `10` | ✓ agree | ✗ differ | q2 is the odd one out | Apply $X_2$ |

**Intuition for `11`**: if q0 and q1 disagree *and* q1 and q2 also disagree, then q1 must be the flipped qubit — it is the common element in both mismatches, the one that looks wrong to both of its neighbours. This is exactly classical majority voting, but executed without ever reading the actual logical value.

---

### 6. The Stabilizer Formalism (The Professional Way to Think About This)

Every modern QEC code is described using **stabilizers** — Pauli operators that leave all valid codewords unchanged (eigenvalue $+1$).

The 3-qubit bit-flip code is stabilized by:
$$S_1 = Z_0 Z_1, \qquad S_2 = Z_1 Z_2$$

You can verify:
$$S_1 (\alpha|000\rangle + \beta|111\rangle) = \alpha(+1)(+1)|000\rangle + \beta(-1)(-1)|111\rangle = \alpha|000\rangle + \beta|111\rangle \quad \checkmark$$

When error $X_i$ occurs, it **anti-commutes** with certain stabilizers (since $XZ = -ZX$), flipping their eigenvalue from $+1$ to $-1$. Measuring the stabilizers thus tells us exactly *which error occurred* without touching the logical state.

This stabilizer framework generalises directly to the **surface code** used in real IBM and Google quantum hardware today.

---

### 7. Why Correction Works: Orthogonal Error Subspaces

The deeper reason correction is possible comes from linear algebra.  
The four possible scenarios (no error, error on q0, q1, or q2) live in **four orthogonal 2-dimensional subspaces** of the full 8-dimensional 3-qubit Hilbert space:

$$\mathcal{H}_8 = \mathcal{C} \oplus \mathcal{E}_0 \oplus \mathcal{E}_1 \oplus \mathcal{E}_2$$

Where:
- $\mathcal{C} = \text{span}\{|000\rangle, |111\rangle\}$ — the **code space** (no error)
- $\mathcal{E}_0 = \text{span}\{|100\rangle, |011\rangle\}$ — error on qubit 0
- $\mathcal{E}_1 = \text{span}\{|010\rangle, |101\rangle\}$ — error on qubit 1
- $\mathcal{E}_2 = \text{span}\{|001\rangle, |110\rangle\}$ — error on qubit 2

The syndrome measurement **projects** us into one of these subspaces without disturbing the state within it. We then apply the appropriate $X$ gate to rotate back into $\mathcal{C}$. The logical information, living inside the subspace, is never disturbed.

---

### 8. What This Code Cannot Do — and the Road Ahead

The 3-qubit bit-flip code is a pedagogical foundation, not a production solution. Its limits are important to understand:

| Limitation | Why it matters |
|-----------|----------------|
| **No phase-flip protection** | A $Z$ error ($|+\rangle \leftrightarrow |-\rangle$) is completely undetected |
| **Single errors only** | Two simultaneous bit-flips cause *miscorrection* — the syndrome becomes ambiguous |
| **Not fault-tolerant** | The syndrome measurement circuit itself can introduce errors |
| **Not universal** | You cannot run arbitrary algorithms in the encoded space without additional structure |

**The progression of QEC codes:**
- **Shor's 9-qubit code (1995)** — first complete QEC code; concatenates a bit-flip code with a phase-flip code to handle both error types
- **Steane [[7,1,3]] code (1996)** — encodes 1 logical qubit in 7 physical qubits; corrects both bit and phase flips using the structure of the classical Hamming code
- **Surface codes (2000s–present)** — the leading candidate for real hardware; scales to thousands of physical qubits per logical qubit with continuously running error correction

Each of these is a direct generalisation of exactly the ideas demonstrated in this notebook.

---

> **Key Takeaway**: The 3-qubit bit-flip code works because entanglement allows us to distribute logical information such that errors become *detectable and localizable* without ever revealing the protected information. The syndrome is a fingerprint of the error, not of the data. Every advanced QEC scheme in use today — Steane, surface, color codes — is a sophisticated extension of exactly this principle.

In [ ]:
# Install required packages
!pip install qiskit qiskit-aer qiskit-ibm-runtime pylatexenc

In [2]:
# Imports
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
import random
from IPython.display import display

In [9]:
# Build the Circuit

def build_bit_flip_circuit(apply_error=True, error_qubit=1):
    """
    Builds a 3-qubit bit-flip error detection circuit.

    Parameters:
        apply_error (bool): Whether to introduce an error
        error_qubit (int): Which qubit to apply X error to

    Returns:
        QuantumCircuit: Constructed circuit
    """

    # 3 data qubits + 2 ancilla qubits, 2 classical bits
    qc = QuantumCircuit(5, 2)

    # --- Step 1: Prepare initial state |1⟩ ---
    qc.x(0)

    # --- Step 2: Encode (|1⟩ → |111⟩) ---
    qc.cx(0, 1)
    qc.cx(0, 2)

    # --- Step 3: Introduce error ---
    if apply_error:
        qc.x(error_qubit)

    # --- Step 4: Syndrome measurement ---
    # First syndrome bit
    qc.cx(0, 3)
    qc.cx(1, 3)

    # Measure ancilla qubits
    qc.measure(3, 0)

    # Second syndrome bit
    qc.cx(1, 4)
    qc.cx(2, 4)

    # Measure ancilla qubits
    qc.measure(4, 1)

    return qc

In [ ]:
# Visualize Circuit

qc = build_bit_flip_circuit(apply_error=True, error_qubit=1)
qc.draw('mpl')

In [ ]:
# Run Simulation

simulator = AerSimulator()

result = simulator.run(qc, shots=1000).result()
counts = result.get_counts()

print("Measurement Results:", counts)
fig = plot_histogram(counts)
display(fig)

## Understanding the Results

The 2-bit output corresponds to the syndrome:

| Syndrome | Meaning |
|--------|--------|
| 00 | No error |
| 01 | Error on qubit 0 |
| 11 | Error on qubit 1 |
| 10 | Error on qubit 2 |

If we injected an error on qubit 1, we expect to see `11`.

In [ ]:
# Experiment: No Error vs Error

# No error case
qc_no_error = build_bit_flip_circuit(apply_error=False)

result_no_error = simulator.run(qc_no_error, shots=1000).result()
counts_no_error = result_no_error.get_counts()

# Error case
qc_error = build_bit_flip_circuit(apply_error=True, error_qubit=1)

result_error = simulator.run(qc_error, shots=1000).result()
counts_error = result_error.get_counts()

# Plot comparison
fig = plot_histogram([counts_no_error, counts_error],
               legend=['No Error', 'With Error'])
display(fig)

In [ ]:
# Experiment: Random Errors
# Launch this several times and you will see the end states 00, 01, 10, 11 randomly

def random_error_circuit(error_probability=0.3):
    qc = QuantumCircuit(5, 2)

    # Encode |1⟩
    qc.x(0)
    qc.cx(0, 1)
    qc.cx(0, 2)

    # Apply random error
    if random.random() < error_probability:
        error_qubit = random.choice([0, 1, 2])
        qc.x(error_qubit)

    # --- Step 4: Syndrome measurement ---
    # First syndrome bit
    qc.cx(0, 3)
    qc.cx(1, 3)

    # Second syndrome bit
    qc.cx(1, 4)
    qc.cx(2, 4)

    # Measure ancilla qubits
    qc.measure(3, 0)
    qc.measure(4, 1)

    return qc

qc_random = random_error_circuit()

result_random = simulator.run(qc_random, shots=1000).result()
counts_random = result_random.get_counts()

fig = plot_histogram(counts_random)
display(fig)